In [ ]:
import html
import sys
import logging
import os

sys.path.append("../..")

from pneuma_seeker.provenance.graph import ProvenanceGraph, ProvenanceNode
from pneuma_seeker.core.ir_system.data_model import RetrieverType
from pneuma_seeker.utils.logger import setup_logger

from pyvis.network import Network

In [ ]:
logger = setup_logger(
    name="pneuma_seeker_logger",
    log_path=os.path.join(".", "log"),
    level=logging.INFO,
    max_bytes=10_000_000,
    backup_count=5,
)
prov_graph = ProvenanceGraph(logger)
ext_node = ProvenanceNode(
    output_data_id="user_uploaded_data",
    output_data_ref="",
    source_retriever=RetrieverType.USER,
    op_description="External user-uploaded data",
)
prov_graph.add_node(ext_node, True)
pn_node = ProvenanceNode(
    output_data_id="pneuma_retrieved_data",
    output_data_ref="tables/something.csv",
    source_retriever=RetrieverType.PNEUMA,
    op_description="Pneuma-retrieved tables",
)
prov_graph.add_node(pn_node, True)
c1_node = ProvenanceNode(
    output_data_id="final_table",
    output_data_ref="",
    source_retriever=RetrieverType.MATERIALIZER,
    op_description="Computation 1",
)
prov_graph.add_node(c1_node, True)
prov_graph.connect(ext_node, c1_node)
prov_graph.connect(pn_node, c1_node)

In [ ]:
def visualize_graph_interactive(provenance_graph: ProvenanceGraph):
    net = Network(notebook=True, directed=True, cdn_resources="in_line")

    for node in provenance_graph.nodes.values():
        if node.id not in net.get_nodes():
            tooltip = f"""
            Output Data ID: {html.escape(node.output_data_id)}
            Source: {html.escape(str(node.source_retriever.value))}
            Description: {html.escape(node.op_description)}
            # Children: {len(node.children)}
            # Parents: {len(node.parents)}
            """

            net.add_node(
                node.id,
                label=node.output_data_id,
                title=tooltip,
            )

        for child in node.children:
            if child.id not in net.get_nodes():
                tooltip = f"""
                Output Data ID: {html.escape(child.output_data_id)}
                Source: {html.escape(str(child.source_retriever.value))}
                Description: {html.escape(child.op_description)}
                # Children: {len(child.children)}
                # Parents: {len(child.parents)}
                """
                net.add_node(
                    child.id,
                    label=child.output_data_id,
                    title=tooltip,
                )
            net.add_edge(node.id, child.id)

    net.show("graph.html")
    return net.generate_html()


visualize_graph_interactive(
    prov_graph
)

# UNK

In [ ]:
# import json
# import html
# from typing import Any

# def _node_label(node) -> str:
#     # Short label for node
#     return node.description or str(node.data_ref)

# def _node_title(node) -> str:
#     # Hover tooltip (escaped)
#     src = getattr(node.source_retriever, "name", str(node.source_retriever))
#     return html.escape(f"data_ref: {node.data_ref}\nsource: {src}\nnode_id: {node.id}")

# def visualize_with_visjs(provenance_graph, output_path="provenance_graph.html"):
#     # Build nodes and edges lists
#     nodes = []
#     edges = []
#     for node in provenance_graph.nodes.values():
#         nodes.append({
#             "id": node.id,
#             "label": _node_label(node),
#             "title": _node_title(node),
#         })
#         for child in node.children:
#             edges.append({"from": node.id, "to": child.id})

#     nodes_json = json.dumps(nodes)
#     edges_json = json.dumps(edges)

#     html_template = f"""<!doctype html>
# <html>
# <head>
#   <meta charset="utf-8" />
#   <title>Provenance Graph</title>
#   <style>
#     html, body {{ width:100%; height:100%; margin:0; padding:0; }}
#     #mynetwork {{ width:100%; height:100vh; border: 1px solid #ddd; }}
#   </style>
#   <!-- vis-network CDN -->
#   <script src="https://unpkg.com/vis-network/standalone/umd/vis-network.min.js"></script>
# </head>
# <body>
#   <div id="mynetwork"></div>
#   <script>
#     const nodes = new vis.DataSet({nodes_json});
#     const edges = new vis.DataSet({edges_json});
#     const container = document.getElementById('mynetwork');
#     const data = {{ nodes: nodes, edges: edges }};

#     const options = {{
#       layout: {{
#         hierarchical: {{
#           enabled: true,
#           levelSeparation: 160,
#           nodeSpacing: 120,
#           treeSpacing: 200,
#           direction: 'UD',   // 'UD' (up-down) or 'LR' (left-right)
#           sortMethod: 'directed'
#         }}
#       }},
#       edges: {{
#         arrows: {{ to: {{ enabled: true }} }},
#         smooth: false
#       }},
#       physics: {{ enabled: false }}, // disable physics to avoid wiggly behavior
#       interaction: {{
#         hover: true,
#         navigationButtons: true,
#         keyboard: true
#       }}
#     }};

#     const network = new vis.Network(container, data, options);

#     // show node details in console on click (example)
#     network.on("selectNode", function(params) {{
#       const id = params.nodes[0];
#       const node = nodes.get(id);
#       console.log("Selected node:", node);
#     }});
#   </script>
# </body>
# </html>
# """
#     with open(output_path, "w", encoding="utf-8") as f:
#         f.write(html_template)

#     print(f"Wrote interactive provenance HTML to: {output_path}")
# visualize_with_visjs(prov_graph)